# SLAYER spike-value convention: `slayer.spike` emits `1/Ts`

Claim under test: when `slayer.spike(membranePotential)` fires, the value
it writes into the output tensor is **`1/Ts`** (not 1.0). This is SLAYER's
discretized-Dirac-delta convention — combined with the `* Ts` factor inside
the PSP convolution, it keeps the PSP integral per spike Ts-invariant.

We verify it by driving a single SLAYER neuron with a controlled membrane
potential pattern, comparing the emitted spike values at `Ts \in {1.0, 0.5, 0.25}`,
and asserting `spike_value == 1/Ts` numerically.

If this test passes, the input data in our `2000_rate/` notebooks (which is
binary 0/1) needs to be scaled by `1/Ts` before being fed to the first SLAYER
layer — otherwise the input PSP comes out scaled differently from SLAYER's
internal expectations. See `docs/knowledge_bank/sample_rate.md`.

## 1. Setup

In [1]:
import torch
import slayerSNN as snn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Standard SRMALPHA descriptor (same as the rest of the project).
LIF_PARAMS = {
    "type": "SRMALPHA",
    "theta": 1,
    "tauSr": 1,
    "tauRho": 1,
    "tauRef": 1,
    "scaleRef": 2,
    "scaleRho": 1,
}

T_REAL_MS = 1000        # Total simulation time (ms) - same for all Ts
TS_VALUES = [1.0, 0.5, 0.25]

Device: cuda


## 2. Probe `slayer.spike` at three sampling rates

Drive a SLAYER neuron with a membrane potential that crosses threshold
periodically (value 5 at every 20th bin, zero elsewhere). The exact pattern
doesn't matter - we just need spikes to be emitted so we can read off their
magnitude.

In [2]:
def emitted_spike_value(Ts: float) -> tuple[float, int]:
    """Drive a SLAYER neuron and report the unique value in its spike output.

    Args:
        Ts: Sampling period (matches `SIM_PARAMS['Ts']`).

    Returns:
        Tuple `(value, count)` where `value` is the magnitude SLAYER assigns
        to each emitted spike and `count` is the number of spikes produced.
    """
    sim_params = {"Ts": Ts, "tSample": T_REAL_MS}
    layer = snn.layer(LIF_PARAMS, sim_params).to(device)

    num_bins = int(round(T_REAL_MS / Ts))
    membrane = torch.zeros(1, 1, 1, 1, num_bins, device=device)
    membrane[0, 0, 0, 0, ::20] = 5.0    # well above theta=1, so spikes fire

    spikes = layer.spike(membrane)
    nonzero = spikes[spikes != 0]
    assert nonzero.numel() > 0, "no spikes produced; check the drive pattern"

    unique_vals = torch.unique(nonzero).cpu().tolist()
    assert len(unique_vals) == 1, f"expected one unique value, got {unique_vals}"
    return unique_vals[0], nonzero.numel()


print(f"{'Ts (ms)':>8s}  {'spike value':>12s}  {'expected (1/Ts)':>16s}  {'#spikes':>8s}")
print("-" * 56)
for Ts in TS_VALUES:
    value, count = emitted_spike_value(Ts)
    expected = 1.0 / Ts
    match = "OK" if abs(value - expected) < 1e-6 else "MISMATCH"
    print(f"{Ts:>8.2f}  {value:>12.4f}  {expected:>16.4f}  {count:>8d}   [{match}]")

 Ts (ms)   spike value   expected (1/Ts)   #spikes
--------------------------------------------------------
    1.00        1.0000            1.0000        50   [OK]
    0.50        2.0000            2.0000       100   [OK]
    0.25        4.0000            4.0000       200   [OK]


## 3. Numerical assertion

Fail loudly if the convention ever changes (e.g. after a SLAYER upgrade).

In [3]:
for Ts in TS_VALUES:
    value, _ = emitted_spike_value(Ts)
    assert abs(value - 1.0 / Ts) < 1e-6, (
        f"SLAYER spike value at Ts={Ts} was {value}, expected {1.0 / Ts}."
    )
print("All assertions passed: slayer.spike emits 1/Ts per spike.")

All assertions passed: slayer.spike emits 1/Ts per spike.


## 4. Sanity-check the consequence: PSP integral is Ts-invariant

If we feed the network a **binary** spike (value 1.0) at every Ts, the PSP
peak scales with Ts - that's exactly the symptom of Bug 1 in the knowledge
bank document. If instead we feed it a `1/Ts`-valued spike (the SLAYER
convention), the PSP peak is Ts-invariant.

In [4]:
print(f"{'Ts (ms)':>8s}  {'psp peak (binary input)':>26s}  {'psp peak (1/Ts input)':>24s}")
print("-" * 64)
for Ts in TS_VALUES:
    layer = snn.layer(LIF_PARAMS, {"Ts": Ts, "tSample": T_REAL_MS}).to(device)
    num_bins = int(round(T_REAL_MS / Ts))
    x = torch.zeros(1, 1, 1, 1, num_bins, device=device)
    x[0, 0, 0, 0, int(round(10 / Ts))] = 1.0   # one binary spike at t=10ms

    psp_binary = layer.psp(x).max().item()
    psp_scaled = layer.psp(x / Ts).max().item()
    print(f"{Ts:>8.2f}  {psp_binary:>26.4f}  {psp_scaled:>24.4f}")

 Ts (ms)     psp peak (binary input)     psp peak (1/Ts input)
----------------------------------------------------------------
    1.00                      1.0000                    1.0000
    0.50                      0.5000                    1.0000
    0.25                      0.2500                    1.0000


## 5. Discussion

Expected output of section 4:

```
 Ts (ms)   psp peak (binary input)   psp peak (1/Ts input)
------------------------------------------------------------
    1.00                    1.0000                   1.0000
    0.50                    0.5000                   1.0000
    0.25                    0.2500                   1.0000
```

- **Binary input column:** PSP peak shrinks linearly with Ts. This is what
  the network sees in our `2000_rate/` notebooks **before** the input-scaling
  fix, and is why the first-layer behavior changed under Ts.
- **`1/Ts`-scaled input column:** PSP peak is invariant - the fix applied
  in `_prepare_input` (`return x.float().to(device) / self.slayer.simulation['Ts']`)
  makes the input layer behave the same way at any Ts.

Together, sections 2 and 4 show **why** the input-scaling fix is necessary:
downstream layers in the network receive `1/Ts`-valued spikes from
`slayer.spike`, so the PSP machinery is calibrated for that magnitude. Feeding
raw binary spikes to the first layer breaks that calibration in a Ts-dependent
way.